# Does seed expansion find the members it was not given?


**The test (instruction 64.12).** Half of a niche's hyperLOPIT markers, whole orthogroups at a time, are given as seeds; the other half are hidden. Seed expansion (random walk with restart over the measured networks and the measurement space) ranks the genome with the hyperLOPIT compartment excluded, and the hidden members are counted at fixed depths. The candidate universe is every gene except the seeds. **What this is not:** the genes that are not markers are not verified negatives, so precision here is a lower bound on how often a ranked gene is truly in the niche.

Built 2026-11-07.

In [1]:
import json, os, subprocess, sys
from concurrent.futures import ThreadPoolExecutor
import pandas as pd
NICHES = ['rhoptries', 'micronemes', 'dense granules', 'IMC', 'apicoplast', 'mitochondrion']
STRATEGIES = ['seed_expansion', 'positive_unlabeled']
SCRIPT = os.path.join(REPO, 'scripts', 'build_seed_expansion_benchmark_notebook.py')
def run(job):
    strategy, niche = job
    tag = niche.replace(' ', '_')
    out = os.path.join(OUT, f'row_{tag}.json' if strategy == 'seed_expansion' else f'row_{strategy}_{tag}.json')
    if not os.path.exists(out):      # a finished niche is reused, never silently recomputed
        subprocess.run([sys.executable, SCRIPT, '--niche', niche, '--out', out, '--strategy', strategy],
                       check=True, env={**os.environ, 'OMP_NUM_THREADS': '2'})
    row = json.load(open(out)); row.setdefault('strategy', strategy)
    return row
jobs = [(s, n) for s in STRATEGIES for n in NICHES]
with ThreadPoolExecutor(max_workers=len(NICHES)) as pool:
    rows = list(pool.map(run, jobs))
table = pd.DataFrame(rows)
table.to_csv(os.path.join(OUT, 'recovery.tsv'), sep='\t', index=False)
table.pivot(index='niche', columns='strategy', values='recall_at_100')

strategy,positive_unlabeled,seed_expansion
niche,,
IMC,0.474,0.526
apicoplast,0.458,0.292
dense granules,0.762,0.571
micronemes,0.786,0.857
mitochondrion,0.371,0.343
rhoptries,0.769,0.808
